<a href="https://colab.research.google.com/github/AnthonyMath1022/AnthonyMath1022/blob/main/Retail_Analysis_ChebConvTCN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display
from google.colab import drive
drive.mount('/content/drive')
%load_ext cudf.pandas
print('Loading data...')
train_path = '/content/drive/MyDrive/Reatilforecast/src/train.csv'
train_df = pd.read_csv(train_path, low_memory=True)
train_df['date'] = pd.to_datetime(train_df['date'])
pd.set_option('display.max_columns', 10)
display(train_df)


Mounted at /content/drive
Loading data...


/tmp/ipython-input-2450478014.py:9: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(train_path, low_memory=True)


,id,date,store_nbr,item_nbr,unit_sales,onpromotion
0,0,2013-01-01,25,103665,7.0,NaN
1,1,2013-01-01,25,105574,1.0,NaN
2,2,2013-01-01,25,105575,2.0,NaN
3,3,2013-01-01,25,108079,1.0,NaN
4,4,2013-01-01,25,108701,1.0,NaN
...,...,...,...,...,...,...
125497035,125497035,2017-08-15,54,2089339,4.0,False
125497036,125497036,2017-08-15,54,2106464,1.0,True
125497037,125497037,2017-08-15,54,2110456,192.0,False
125497038,125497038,2017-08-15,54,2113914,198.0,True


In [2]:
start_date = '2017-07-01'
end_date = '2017-12-31'

train_df = train_df[(train_df['date'] >= start_date) & (train_df['date'] <= end_date)]
pd.set_option('display.max_columns', 10)
display(train_df)
train_df.count()
train_df.isna().sum()


,id,date,store_nbr,item_nbr,unit_sales,onpromotion
120642911,120642911,2017-07-01,1,99197,2.0,False
120642912,120642912,2017-07-01,1,103520,2.0,False
120642913,120642913,2017-07-01,1,103665,11.0,False
120642914,120642914,2017-07-01,1,105574,2.0,False
120642915,120642915,2017-07-01,1,105575,3.0,False
...,...,...,...,...,...,...
125497035,125497035,2017-08-15,54,2089339,4.0,False
125497036,125497036,2017-08-15,54,2106464,1.0,True
125497037,125497037,2017-08-15,54,2110456,192.0,False
125497038,125497038,2017-08-15,54,2113914,198.0,True


,0
id,0
date,0
store_nbr,0
item_nbr,0
unit_sales,0
onpromotion,0


In [3]:
import pandas as pd

item_path = '/content/drive/MyDrive/Reatilforecast/src/items.csv'
store_path = '/content/drive/MyDrive/Reatilforecast/src/stores.csv'
oil_path = '/content/drive/MyDrive/Reatilforecast/src/oil.csv'

item = pd.read_csv(item_path)
store = pd.read_csv(store_path)
oil = pd.read_csv(oil_path )
oil = oil[(oil['date'] >= start_date) & (oil['date'] <= end_date)]

train_df = pd.merge(train_df,item,on='item_nbr',how='left')
train_df = pd.merge(train_df,store,on='store_nbr',how='left')
train_df.isna().sum()

id             0
date           0
store_nbr      0
item_nbr       0
unit_sales     0
onpromotion    0
family         0
class          0
perishable     0
city           0
state          0
type           0
cluster        0
dtype: int64

In [4]:
import numpy as np

def most_frequent_sales(data, variable, N, all='TRUE'):


    # counts
    values, counts = np.unique(data[variable].to_numpy(), return_counts=True)

    # Ensure values and counts are 2D column vectors before stacking
    values_2d = values.reshape(-1, 1)
    counts_2d = counts.reshape(-1, 1)

    # Nx2 table: [value, count]
    labels_freq_pd = np.column_stack((values_2d, counts_2d))

    # sort by count desc
    labels_freq_pd = labels_freq_pd[np.argsort(labels_freq_pd[:, 1])[::-1]]

    # keep top N
    topN = labels_freq_pd[:N]
    main_labels = topN[:, 0] if all == 'False' else labels_freq_pd[:, 0]

    # raw labels (replace deprecated as_matrix)
    labels_raw_np = data[variable].to_numpy().reshape(-1, 1)

    # indices where label in main_labels (same style as your labels_filtered_index[0])
    labels_filtered_index = np.where(np.isin(labels_raw_np.ravel(), main_labels))

    return topN, labels_filtered_index

label_freq, labels_filtered_index = most_frequent_sales(train_df, 'item_nbr', 5, 'FALSE')
print("pd_train.shape=", labels_filtered_index[0].shape)

pd_train_filtered = train_df.loc[labels_filtered_index[0], :]
print("pd_train_filtered.shape = ", pd_train_filtered.shape)


pd_train.shape= (4854129,)
pd_train_filtered.shape =  (4854129, 13)


In [5]:
pd_train_filtered  = pd_train_filtered.drop(['city','state','type','cluster','store_nbr','item_nbr','family','class','id'], axis = 1)

In [6]:

dummy_variables = ['onpromotion','perishable']

for var in dummy_variables:
    dummy = pd.get_dummies(pd_train_filtered [var], prefix = var, drop_first = False).astype(int)
    pd_train_filtered  = pd.concat([pd_train_filtered ,dummy], axis = 1)

pd_train_filtered  = pd_train_filtered.drop(dummy_variables, axis = 1)


In [7]:
import pandas as pd

# 1. Ensure your date columns are actual datetime objects
pd_train_filtered ['date'] = pd.to_datetime(pd_train_filtered ['date'])
oil['date'] = pd.to_datetime(oil['date'])

# 2. Generate the complete date range automatically
# pd.date_range replaces the manual loop and delta calculation
calendar = pd.DataFrame({
    'date': pd.date_range(start=train_df.date.min(), end=train_df.date.max())
})

# 3. Merge
oil = calendar.merge(oil, on='date', how='left')

In [8]:
na_index_oil = oil[oil['dcoilwtico'].isnull() == True].index.values

#Define the index to use to apply the formala
na_index_oil_plus = na_index_oil.copy()
na_index_oil_minus = np.maximum(0, na_index_oil-1)

for i in range(len(na_index_oil)):
    k = 1
    while (na_index_oil[min(i+k,len(na_index_oil)-1)] == na_index_oil[i]+k):
        k += 1
    na_index_oil_plus[i] = min(len(oil)-1, na_index_oil_plus[i] + k )

#Apply the formula
for i in range(len(na_index_oil)):
    if (na_index_oil[i] == 0):
        oil.loc[na_index_oil[i], 'dcoilwtico'] = oil.loc[na_index_oil_plus[i], 'dcoilwtico']
    elif (na_index_oil[i] == len(oil)):
        oil.loc[na_index_oil[i], 'dcoilwtico'] = oil.loc[na_index_oil_minus[i], 'dcoilwtico']
    else:
        oil.loc[na_index_oil[i], 'dcoilwtico'] = (oil.loc[na_index_oil_plus[i], 'dcoilwtico'] + oil.loc[na_index_oil_minus[i], 'dcoilwtico'])/ 2

pd_train_filtered  = pd_train_filtered.merge(oil, left_on='date', right_on='date', how='left')


In [9]:
pd_train_filtered.sample(10)

,date,unit_sales,onpromotion_False,onpromotion_True,perishable_0,perishable_1,dcoilwtico
2533073,2017-07-24,1.0,1,0,1,0,46.21
1354975,2017-07-13,3.0,1,0,1,0,46.06
322661,2017-07-03,47.0,1,0,0,1,45.11
2585065,2017-07-25,16.0,1,0,1,0,47.77
273615,2017-07-03,15.0,0,1,1,0,45.11
2487104,2017-07-24,3.0,1,0,1,0,46.21
1124408,2017-07-11,4.0,1,0,1,0,45.06
2797593,2017-07-27,2.0,1,0,1,0,49.05
599147,2017-07-06,1.0,1,0,1,0,45.52
1831296,2017-07-18,2.0,1,0,0,1,46.40


In [10]:
pd_train_filtered.count()

date                 4854129
unit_sales           4854129
onpromotion_False    4854129
onpromotion_True     4854129
perishable_0         4854129
perishable_1         4854129
dcoilwtico           4854129
dtype: int64

In [11]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# --- CONFIGURATION ---
LOOKBACK = 7
HORIZON = 1
TOP_N = 10

# 1. Filter for Top 10 Items
# We use a wider date range (from Jan 2017) to ensure enough data for lookback and splitting.
local_start_date = '2017-01-01'

temp_base_df = pd.read_csv(train_path, low_memory=True)
temp_base_df['date'] = pd.to_datetime(temp_base_df['date'])
temp_base_df = temp_base_df[(temp_base_df['date'] >= local_start_date) & (temp_base_df['date'] <= end_date)].copy()

# Re-merge with item and store data
temp_item_df = pd.read_csv(item_path)
temp_store_df = pd.read_csv(store_path)
temp_base_df = pd.merge(temp_base_df, temp_item_df, on='item_nbr', how='left')
temp_base_df = pd.merge(temp_base_df, temp_store_df, on='store_nbr', how='left')

# Reload and merge Oil data (full range to cover Jan-July)
oil_full = pd.read_csv(oil_path)
oil_full['date'] = pd.to_datetime(oil_full['date'])
temp_base_df = pd.merge(temp_base_df, oil_full[['date', 'dcoilwtico']], on='date', how='left')

# Top 10 items
top_items = temp_base_df['item_nbr'].value_counts().head(TOP_N).index
pd_train_filtered = temp_base_df[temp_base_df['item_nbr'].isin(top_items)].copy()

# 2. Pivot the Data to Wide Format
# Sales Pivot
pivoted_sales = pd_train_filtered.pivot_table(
    index='date',
    columns='item_nbr',
    values='unit_sales',
    fill_value=0
)
pivoted_sales.columns = [f'sales_{i}' for i in pivoted_sales.columns]

# Promotion Pivot
if 'onpromotion' in pd_train_filtered.columns:
    pd_train_filtered['onpromotion'] = pd_train_filtered['onpromotion'].astype(int)
    pivoted_promo = pd_train_filtered.pivot_table(
        index='date',
        columns='item_nbr',
        values='onpromotion',
        fill_value=0
    )
    pivoted_promo.columns = [f'promo_{i}' for i in pivoted_promo.columns]
else:
    pivoted_promo = pd.DataFrame()

# Oil Data (Common Feature)
oil_series = pd_train_filtered.groupby('date')['dcoilwtico'].max()

# 3. Combine into one Wide Dataframe
combined = pd.concat([pivoted_sales, pivoted_promo, oil_series], axis=1)

# Fill gaps
combined = combined.ffill().bfill()

# Define targets and features
stock_cols = pivoted_sales.columns.tolist()
print(f"Target Columns ({len(stock_cols)}): {stock_cols}")
print(f"Total Features: {combined.shape[1]}")
print(f"Total Data Points (Days): {len(combined)}")

# --- TRAIN/TEST SPLIT ---
# Adjusted test_len to 16 days (typical for this dataset's horizon)
'''
test_len = 30
trainval = combined.iloc[:-test_len]
test_data = combined.iloc[-test_len:]

val_len = int(len(trainval) * 0.2)
train_len = len(trainval) - val_len

train_data = trainval.iloc[:train_len]
val_data = trainval.iloc[train_len:]

print(f"Train size: {len(train_data)}, Val size: {len(val_data)}, Test size: {len(test_data)}")

# Scaling
feat_scaler = MinMaxScaler()
feat_scaler.fit(train_data.values)

train_scaled = feat_scaler.transform(train_data.values)
val_scaled = feat_scaler.transform(val_data.values)
test_scaled = feat_scaler.transform(test_data.values)
'''

/usr/local/lib/python3.12/dist-packages/cudf/pandas/fast_slow_proxy.py:28: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  return fn(*args, **kwargs)


Target Columns (10): ['sales_222879', 'sales_261052', 'sales_265559', 'sales_314384', 'sales_323013', 'sales_364606', 'sales_502331', 'sales_1052563', 'sales_1157564', 'sales_1162382']
Total Features: 21
Total Data Points (Days): 227
Train size: 158, Val size: 39, Test size: 30


In [12]:
corr = combined[stock_cols].corr(method='spearman')
corr = (corr - (corr==1))
corr = corr.abs()
corr.head(10)
corr_matrix = corr

In [13]:
from sklearn.feature_selection import mutual_info_regression

def mutual_info_matrix(data):
    n = data.shape[1]
    mi_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i != j:
                mi = mutual_info_regression(
                    data[:, [i]], data[:, j]
                )[0]
                mi_matrix[i, j] = mi
    return mi_matrix

mi_matrix = mutual_info_matrix(combined[stock_cols].values).round(4)

In [14]:
def create_combine_edge(corr_matrix, mi_matrix, corr_threshold=0.3, mi_threshold=0.1):
    edge_indices = []
    edge_weights = []

    for i in range(len(corr_matrix)):
        for j in range(len(corr_matrix)):
            if i !=j:
                corr_val = abs(corr_matrix.iloc[i, j])
                mi_val = abs(mi_matrix[i, j])
                if corr_val > corr_threshold and mi_val > mi_threshold:
                    edge_indices.append((i, j))
                    edge_weights.append((corr_val, mi_val))
    return edge_indices, edge_weights

edge_indices, edge_weights = create_combine_edge(corr_matrix, mi_matrix, corr_threshold=0.3, mi_threshold=0.1)
node_features = torch.tensor(combined[stock_cols].values.T, dtype=torch.float)

NameError: name 'torch' is not defined

In [ ]:
from torch_geometric.data import Data
edge_indices, edge_weights = create_combine_edge(corr_matrix, mi_matrix, corr_threshold=0.3, mi_threshold=0.1)
node_features = torch.tensor(combined[stock_cols].values.T, dtype=torch.float)
edge_index_t = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()
edge_index = torch.cat([edge_index_t, edge_index_t[[1, 0], :]], dim=1)
edge_attr = torch.tensor(edge_weights, dtype=torch.float)
edge_attr = torch.cat([edge_attr, edge_attr], dim=0)

target = torch.tensor(combined[stock_cols].values.T, dtype=torch.float)
stock_graph_data = Data(x=node_features, edge_index=edge_index, edge_attr=edge_attr, y=target)

class StockData(Data):
    def __init__(self, x=None, edge_index=None, edge_attr=None, y=None):
        super(StockData, self).__init__(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)

custom_stock_data = StockData(x=node_features, edge_index=edge_index, edge_attr=edge_attr, y=target)
print(custom_stock_data)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Initialize the scaler
scaler = MinMaxScaler()

# Scale the stock price data
scaled_data = scaler.fit_transform(combined[stock_cols])

# Create a new data object for the normalized data
normalized_data = stock_graph_data.clone()

# Update the node features and targets with the scaled data
# The data needs to be transposed to match the shape [num_nodes, num_timesteps]
normalized_data.x = torch.tensor(scaled_data.T, dtype=torch.float)
normalized_data.y = torch.tensor(scaled_data.T, dtype=torch.float)

print("\\n--- Normalized Data Object ---")
print(normalized_data)
print("\\nNormalized node features (first 5 values of first node):")
print(normalized_data.x[0, :5])

print(f"\\nMin of normalized features for the first node: {torch.min(normalized_data.x[0])}")
print(f"Max of normalized features for the first node: {torch.max(normalized_data.x[0])}")

In [ ]:
from torch_geometric_temporal.signal import StaticGraphTemporalSignal
from torch_geometric_temporal import temporal_signal_split

def make_graph_windows(signal, n_in: int, n_out: int, multi_step: bool = True):
    """
    Rolling window creation for temporal GNN forecasting.

    Args:
        signal: StaticGraphTemporalSignal with T timesteps
        n_in: input sequence length
        n_out: forecast horizon
        multi_step: if True, return [n_out, N, F]; else [N, F]
    """
    feats = [np.asarray(f, dtype=np.float32) for f in signal.features]
    T = len(feats)
    windows = []

    # Calculate valid starting positions
    num_windows = T - n_in - n_out + 1

    if num_windows <= 0:
        raise ValueError(f"Not enough timesteps: T={T}, need at least {n_in + n_out}")

    for i in range(num_windows):
        # Input: timesteps [i, i+n_in)
        X = np.stack(feats[i : i + n_in], axis=0)  # [n_in, N, F]

        # Output: timesteps [i+n_in, i+n_in+n_out)
        if multi_step:
            Y = np.stack(feats[i + n_in : i + n_in + n_out], axis=0)  # [n_out, N, F]
        else:
            Y = feats[i + n_in]  # [N, F]

        windows.append((X, Y))

    return {
        "edge_index": torch.tensor(signal.edge_index, dtype=torch.long),
        "edge_weight": torch.tensor(signal.edge_weight, dtype=torch.float),

        "windows": windows,
    }



# Build temporal_dataset from existing scaled_data and graph (edge_index/edge_attr)
# Each timestep feature is a single scalar per node (scaled close), and target is next-step value.
T = scaled_data.shape[0]
X_list = [scaled_data[t].reshape(-1, 1).astype(np.float32) for t in range(T - 1)]       # [N, 1]
Y_list = [scaled_data[t + 1].reshape(-1, 1).astype(np.float32) for t in range(T - 1)]   # [N, 1]

edge_index_np = edge_index.detach().cpu().numpy() if isinstance(edge_index, torch.Tensor) else np.asarray(edge_index)
# Use the first column of edge_attr (correlation) as edge weights
edge_weight_np = (edge_attr[:, 0].detach().cpu().numpy()
                  if isinstance(edge_attr, torch.Tensor)
                  else np.asarray(edge_attr)[:, 0])


temporal_dataset = StaticGraphTemporalSignal(
    edge_index=edge_index_np,
    edge_weight=edge_weight_np.astype(np.float32),
    features=X_list,
    targets=Y_list,
)


train_signal, test_signal = temporal_signal_split(temporal_dataset, train_ratio=0.8587948874) # for 2018-01-01 split 0.6843, 2012-01-01 split 0.83
total_timesteps = len(temporal_dataset.features)
split_idx = int(total_timesteps*0.10)
train_signal = train_signal[:-split_idx+1]
val_signal  = train_signal[-split_idx:]
test_signal = test_signal


print(f'train_signal: {len(train_signal.features)}, val_signal: {len(val_signal.features)}, test_signal: {len(test_signal.features)}')

n_in, n_out = 14,1

train_windows = make_graph_windows(train_signal, n_in, n_out, multi_step=True)
val_windows  = make_graph_windows(train_signal[:split_idx],  n_in, n_out, multi_step=True)
test_windows  = make_graph_windows(test_signal,  n_in, n_out, multi_step=True)
print(f"# train windows: {len(train_windows['windows'])},# val windows: {len(val_windows['windows'])}, # test windows: {len(test_windows['windows'])}")
